In [ ]:
from fourth_day import Fourth_Day , config
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import config_icetray 
import logging
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import pickle
import glob

In [ ]:
def create_config(config, **kwargs):

    if "rs" in kwargs:
        rs = kwargs["rs"]
    else:
        rs = np.random.randint(0, 10000)
    config['general']['random state seed'] = rs
    config['general']['enable logging'] = False  
    config['general']['debug level'] = logging.DEBUG 
    config['general']['log file handler'] = '/home/clagunas/projects/rpp-nahee/clagunas/fourth_day/run/fd.log'  
    config['general']['config location'] = '/home/clagunas/projects/rpp-nahee/clagunas/fourth_day/run/config.txt' 

    config['scenario']['organism movement'] = False # was false by default
    config['scenario']['population size'] = 10  # The starting population size
    config['scenario']['duration'] = 300 * 1  # Total simulation time in seconds
    config['scenario']['exclusion'] = False  # If an exclusion zone should be used (the detector)
    config['scenario']['injection']['rate'] = 1  #  Injection rate in per second, a distribution is constructed from this value
    config['scenario']['injection']['y range'] = [0., 10.]  # The y-range of injection
    config['scenario']['detector'] = {  # detector specific properties, positions are defined as offsets from the light prop values
        "switch": True,  # If the detector should be modelled
        "type": "POM",  # Detector name, implemented types are given in the config
        "response": True,  # If a detector response should be used
        "acceptance": "Flat",  # Flat acceptance
        "mean detection prob": 1.  # Used for the acceptance calculation
    }

    config['organisms']['emission fraction'] = 0.1  # Amount of energy an organism uses per pulse
    config['organisms']['alpha'] = 1e0  # Proportionality factor for the emission probability
    config['organisms']["minimal shear stress"] = 0.005  # The minimal amount of shear stress needed to emit (generic units)
    config["organisms"]["filter"] = 'depth'  # Method of filtering organisms (here depth)
    config["organisms"]["depth filter"] = 1000.  # Organisms need to exist below this depth
    
    config['water']['model']['name'] = 'custom'  
    config['water']['model']['off set'] = np.array([0., 0.])  
    config['water']['model']['directory'] = "/Parabola_5mm/run_5cm_npy/"  
    config['water']['model']['time step'] = 1.  

    return config

In [ ]:
def plot_water_velocity_and_organisms(fd, time_step):
   
    df_step = fd.statistics[time_step]
    
    x = np.arange(0,15,0.01) #0,10
    y = np.arange(5,10,0.01) #5,10
    points = np.asarray( [ [i, j] for i in x for j in y ] )
    res = fd._current.velocities(points, out_nr = 100)

    X, Y = np.meshgrid(x, y, indexing='ij')
    Z = res[-1].reshape(len(x), len(y))

    fig, ax = plt.subplots()

    pc = ax.pcolormesh(X, Y, Z, shading='auto')
    #fig.colorbar(pc, ax=ax)

    ax.scatter(df_step.loc[~df_step['is_emitting'],'pos_x'], 
            df_step.loc[~df_step['is_emitting'], 'pos_y'], 
            c='red', s=5)
    ax.scatter(df_step.loc[df_step['is_emitting'],'pos_x'], 
            df_step.loc[df_step['is_emitting'], 'pos_y'], 
            c='yellow', s=5)

    ax.set_xlim(0, 15)
    ax.set_ylim(5, 10)
    ax.set_aspect('equal', adjustable='box')

    plt.show()  

In [ ]:
def animate_water_velocity_and_organisms(fd, time_steps, interval=200, saveas=None):
    """
    fd           your simulation object
    time_steps   list or range of time indices
    interval     ms between frames
    """
  
    x = np.arange(0, 15, 0.01)   
    y = np.arange(5, 10, 0.01)

    X, Y = np.meshgrid(x, y, indexing='ij')

    fig, ax = plt.subplots(figsize=(6, 4))
    points = np.column_stack([X.ravel(), Y.ravel()])
    res = fd._current.velocities(points, out_nr=time_steps[0])
    Z0 = res[-1].reshape(len(x), len(y))

    pc = ax.pcolormesh(X, Y, Z0, shading='auto')

    # placeholders
    scat_emit = ax.scatter([], [], c='yellow', s=5)
    scat_no_emit = ax.scatter([], [], c='red', s=5)

    ax.set_xlim(0, 15)
    ax.set_ylim(5, 10)
    ax.set_aspect('equal', adjustable='box')

    def update(frame_idx):
        
        t = time_steps[frame_idx]

        points = np.column_stack([X.ravel(), Y.ravel()])
        res = fd._current.velocities(points, out_nr=t)
        Z = res[-1].reshape(X.shape)

        pc.set_array(Z.ravel())

        df_step = fd.statistics[t]

        scat_emit.set_offsets(
            df_step.loc[df_step['is_emitting'], ['pos_x', 'pos_y']].values
        )
        scat_no_emit.set_offsets(
            df_step.loc[~df_step['is_emitting'], ['pos_x', 'pos_y']].values
        )

        ax.set_title(f"Time step: {t}")
        return pc, scat_emit, scat_no_emit

    anim = FuncAnimation(
        fig,
        update,
        frames=len(time_steps),
        interval=interval,
        blit=False
    )

    plt.show()

    if saveas:
        anim.save(saveas, writer="imagemagick")
        
    return anim


In [ ]:
baseconfig = create_config(config_icetray._baseconfig, rs = 42)
fd = Fourth_Day(userconfig=baseconfig)

In [ ]:
fd.sim()

In [ ]:
fd._current._save_string_vel

In [ ]:
plot_water_velocity_and_organisms(fd, time_step=200)

In [ ]:
time_steps = range(0,100) 
anim = animate_water_velocity_and_organisms(fd, time_steps, interval=150, saveas='animation_movement_false_42_5cms.gif')
HTML(anim.to_jshtml())

In [ ]:
baseconfig = create_config(config_icetray._baseconfig, rs = 42)
fd_2 = Fourth_Day(userconfig=baseconfig)
fd_2.sim()

In [ ]:
baseconfig = create_config(config_icetray._baseconfig, rs = 45)
fd_3 = Fourth_Day(userconfig=baseconfig)
fd_3.sim()

In [ ]:
plot_water_velocity_and_organisms(fd, time_step=100)
plot_water_velocity_and_organisms(fd_2, time_step=100)
plot_water_velocity_and_organisms(fd_3, time_step=100)

In [ ]:
fd.statistics[0]

In [ ]:
fd.statistics[-1]

In [ ]:
# fds = []
# for i in tqdm(range(5)):
#     baseconfig = create_config(config_icetray._baseconfig, rs = 42 + i)
#     fd_temp = Fourth_Day(userconfig=baseconfig)
#     fd_temp.sim()
#     fds.append(fd_temp)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import pickle
import glob

In [ ]:
files = sorted(glob.glob("all_simulations_movement_True*.pkl"))

#all_simulations_true = []
time_step = 299
emission_positions_true = []

for f in files:
    print("Loading:", f)
    with open(f, "rb") as handle:
        data = pickle.load(handle)
        #all_simulations_true.extend(data)   # extend, not append!
        # Collect all x,y positions where emission occurs at this time step

        for sim in data:
            df_step = sim.statistics[time_step]
            emitting = df_step.loc[df_step['is_emitting'], ['pos_x', 'pos_y']].to_numpy()
            if len(emitting) > 0:
                #print('emission happened')
                emission_positions_true.append(emitting)

# Stack all positions into a single array
if len(emission_positions_true) > 0:
    emission_positions_true = np.vstack(emission_positions_true)
else:
    emission_positions_true = np.empty((0,2))

In [ ]:
files = sorted(glob.glob("all_simulations_movement_False_seed_*.pkl"))

#all_simulations_true = []
time_step = 299
emission_positions_false = []

for f in files:
    print("Loading:", f)
    with open(f, "rb") as handle:
        data = pickle.load(handle)
        #all_simulations_true.extend(data)   # extend, not append!
        # Collect all x,y positions where emission occurs at this time step

        for sim in data:
            df_step = sim.statistics[time_step]
            emitting = df_step.loc[df_step['is_emitting'], ['pos_x', 'pos_y']].to_numpy()
            if len(emitting) > 0:
                #print('emission happened')
                emission_positions_false.append(emitting)

# Stack all positions into a single array
if len(emission_positions_false) > 0:
    emission_positions_false = np.vstack(emission_positions_false)
else:
    emission_positions_false = np.empty((0,2))

In [ ]:
x_bins = np.linspace(0, 15, 100) 
y_bins = np.linspace(5, 10, 100)

H, xedges, yedges = np.histogram2d(
    emission_positions_true[:,0],
    emission_positions_true[:,1],
    bins=[x_bins, y_bins]
)

H_normalized = H / H.sum()

plt.figure(figsize=(8,4))
plt.imshow(
    H_normalized.T,   
    origin='lower',
    extent=[x_bins[0], x_bins[-1], y_bins[0], y_bins[-1]],
    aspect='equal',
    cmap='hot'
)
#plt.colorbar(label='Normalized emission frequency')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Emission heatmap at time step {time_step} for Movement = True')
plt.show()

In [ ]:
x_bins = np.linspace(0, 15, 100) 
y_bins = np.linspace(5, 10, 100)

H, xedges, yedges = np.histogram2d(
    emission_positions_false[:,0],
    emission_positions_false[:,1],
    bins=[x_bins, y_bins]
)

H_normalized = H / H.sum()

plt.figure(figsize=(8,4))
plt.imshow(
    H_normalized.T,
    origin='lower',
    extent=[x_bins[0], x_bins[-1], y_bins[0], y_bins[-1]],
    aspect='equal',
    cmap='hot'
)
#plt.colorbar(label='Normalized emission frequency')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Emission heatmap at time step {time_step} for Movement = False')
#plt.set_aspect('equal', adjustable='box')
plt.show()